# Expedia Churn Case Study

This notebook keeps the analysis focused on the three case-study expectations:

1. Identify which factors are most associated with churn.
2. Build a statistical model to identify customers/bookings more vs. less likely to churn.
3. Summarize insights and recommendations for an Expedia leadership presentation.

The approach is intentionally practical and interview-case-study friendly: exploratory churn-rate cuts, a simple logistic regression model, clear model evaluation, and business recommendations.

## 1. Setup

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

PROJECT_DIR = Path('/Users/ejazanwar/Desktop/Codex/Expedia_Project')
DATA_PATH = Path('/Users/ejazanwar/Downloads/Case Study/PIP_case_study_data 3 (1).csv')
OUTPUT_DIR = PROJECT_DIR / 'outputs_simple'
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET = 'churn_flag'
DATA_PATH.exists(), DATA_PATH

## 2. Load and Understand the Data

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]:,}')
df.head()

In [ ]:
summary = pd.DataFrame({
    'column': df.columns,
    'dtype': [df[c].dtype for c in df.columns],
    'missing_values': [df[c].isna().sum() for c in df.columns],
    'missing_rate': [df[c].isna().mean() for c in df.columns],
    'unique_values': [df[c].nunique(dropna=True) for c in df.columns],
})
summary

In [ ]:
print('Date range:', df['bk_date'].min(), 'to', df['bk_date'].max())
print('Unique customers:', f"{df['email_address'].nunique():,}")
print('Unique bookings:', f"{df['booking_id'].nunique():,}")
print('Overall churn rate:', f"{df[TARGET].mean():.1%}")

## 3. Clean the Data

Simple cleaning choices:

- Parse booking and cancellation dates.
- Fill missing `cancel_flag` with 0 because missing cancellation status is rare.
- Fill missing `marketing_channel` as `Missing`.
- Fill missing `total_visit_minutes` with the median.
- Exclude `email_address` and `booking_id` because they are identifiers.
- Exclude `cancel_date` from modeling because it may be post-booking information and can create leakage.

In [ ]:
df['bk_date'] = pd.to_datetime(df['bk_date'], errors='coerce')
df['cancel_date'] = pd.to_datetime(df['cancel_date'], errors='coerce')
df['cancel_flag'] = df['cancel_flag'].fillna(0).astype(int)
df['marketing_channel'] = df['marketing_channel'].fillna('Missing')
df['total_visit_minutes'] = df['total_visit_minutes'].fillna(df['total_visit_minutes'].median())

# Simple date features that are available at booking time.
df['booking_month'] = df['bk_date'].dt.month
df['booking_dayofweek'] = df['bk_date'].dt.dayofweek

clean_check = df[['cancel_flag', 'marketing_channel', 'total_visit_minutes', 'bk_date', TARGET]].isna().sum()
clean_check

## 4. Factor Importance from Churn-Rate Cuts

Before modeling, compare churn rates across the provided factors. This gives a clear business view of which factors are associated with higher or lower churn.

In [ ]:
def churn_by_factor(data, factor):
    out = (
        data.groupby(factor, dropna=False)[TARGET]
        .agg(churn_rate='mean', bookings='count')
        .reset_index()
        .sort_values('churn_rate', ascending=False)
    )
    out['churn_rate_pct'] = (out['churn_rate'] * 100).round(1)
    return out

factors_to_review = [
    'customer_type',
    'loyalty_tier',
    'platform',
    'marketing_channel',
    'coupon_flag',
    'pay_now_flag',
    'cancel_flag',
    'hotel_star_rating',
]

factor_tables = {factor: churn_by_factor(df, factor) for factor in factors_to_review}
factor_tables['loyalty_tier']

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
plot_factors = ['customer_type', 'loyalty_tier', 'platform', 'marketing_channel']

for ax, factor in zip(axes.ravel(), plot_factors):
    plot_data = factor_tables[factor].copy()
    plot_data[factor] = plot_data[factor].astype(str)
    sns.barplot(data=plot_data, x='churn_rate', y=factor, ax=ax, color='#2F80ED')
    ax.axvline(df[TARGET].mean(), color='gray', linestyle='--')
    ax.set_title(f'Churn by {factor}')
    ax.set_xlabel('Churn rate')
    ax.set_ylabel('')
    ax.xaxis.set_major_formatter(lambda x, _: f'{x:.0%}')

plt.tight_layout()
plt.show()

In [ ]:
# Summarize factor strength by spread between highest and lowest churn rate.
factor_importance_simple = []
for factor, table in factor_tables.items():
    factor_importance_simple.append({
        'factor': factor,
        'min_churn_rate': table['churn_rate'].min(),
        'max_churn_rate': table['churn_rate'].max(),
        'spread_pp': (table['churn_rate'].max() - table['churn_rate'].min()) * 100,
        'largest_group_bookings': int(table['bookings'].max()),
    })

factor_importance_simple = pd.DataFrame(factor_importance_simple).sort_values('spread_pp', ascending=False)
factor_importance_simple

In [ ]:
plt.figure(figsize=(9, 4.8))
sns.barplot(data=factor_importance_simple, x='spread_pp', y='factor', color='#219653')
plt.title('Simple factor importance: churn-rate spread by factor')
plt.xlabel('Difference between highest and lowest churn rate, percentage points')
plt.ylabel('')
plt.tight_layout()
plt.show()

## 5. Build a Statistical Churn Model

Use logistic regression because it is a standard, explainable statistical model for binary outcomes like churn.

The EDA above uses the full dataset. To keep the notebook fast and easy to rerun, the model is trained on a reproducible 200k-row sample. This is enough for a case-study model and can be changed to full data by setting `model_df = df.copy()`.

In [ ]:
model_features = [
    'coupon_flag',
    'pay_now_flag',
    'cancel_flag',
    'customer_type',
    'loyalty_tier',
    'platform',
    'marketing_channel',
    'total_visit_minutes',
    'total_visit_pages',
    'landing_pages_count',
    'search_pages_count',
    'property_pages_count',
    'bkg_confirmation_pages_count',
    'bounce_visits_count',
    'searched_destinations_count',
    'hotel_star_rating',
    'booking_month',
    'booking_dayofweek',
]

numeric_features = [
    'coupon_flag',
    'pay_now_flag',
    'cancel_flag',
    'loyalty_tier',
    'total_visit_minutes',
    'total_visit_pages',
    'landing_pages_count',
    'search_pages_count',
    'property_pages_count',
    'bkg_confirmation_pages_count',
    'bounce_visits_count',
    'searched_destinations_count',
    'hotel_star_rating',
    'booking_month',
    'booking_dayofweek',
]

categorical_features = ['customer_type', 'platform', 'marketing_channel']

excluded_columns = ['email_address', 'booking_id', 'cancel_date']
print('Excluded from model:', excluded_columns)

model_df = df.sample(n=min(200_000, len(df)), random_state=42).copy()

X = model_df[model_features]
y = model_df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print(f'Modeling sample rows: {len(model_df):,}')
print(f'Train rows: {len(X_train):,}')
print(f'Test rows: {len(X_test):,}')
print(f'Train churn rate: {y_train.mean():.1%}')
print(f'Test churn rate: {y_test.mean():.1%}')

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), numeric_features),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ]
)

model = Pipeline([
    ('preprocessor', preprocessor),
    ('logistic_regression', LogisticRegression(max_iter=300, class_weight='balanced', solver='liblinear')),
])

model.fit(X_train, y_train)

## 6. Evaluate the Model

In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f'ROC-AUC: {roc_auc:.3f}')
print('\nClassification report:')
print(classification_report(y_test, y_pred))

confusion_matrix(y_test, y_pred)

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC-AUC = {roc_auc:.3f}', color='#2F80ED')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.title('Logistic Regression ROC Curve')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Model-Based Driver Importance

For logistic regression, larger absolute coefficients indicate stronger model influence after preprocessing. Positive coefficients increase churn probability; negative coefficients decrease churn probability.

In [ ]:
feature_names = model.named_steps['preprocessor'].get_feature_names_out()
coefs = model.named_steps['logistic_regression'].coef_[0]

coef_table = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs,
    'absolute_coefficient': np.abs(coefs),
    'direction': np.where(coefs > 0, 'Higher churn', 'Lower churn'),
}).sort_values('absolute_coefficient', ascending=False)

coef_table.head(20)

In [ ]:
plot_coef = coef_table.head(15).sort_values('absolute_coefficient')

plt.figure(figsize=(9, 5.5))
sns.barplot(data=plot_coef, x='absolute_coefficient', y='feature', hue='direction', dodge=False)
plt.title('Top model drivers from logistic regression')
plt.xlabel('Absolute coefficient')
plt.ylabel('')
plt.legend(title='Direction')
plt.tight_layout()
plt.show()

## 8. Identify More vs. Less Likely to Churn

Create simple risk groups from predicted churn probability. This turns the model into an actionable business output.

In [ ]:
scored = X_test.copy()
scored['actual_churn'] = y_test.values
scored['predicted_churn_probability'] = y_pred_proba
scored['risk_group'] = pd.qcut(
    scored['predicted_churn_probability'],
    q=[0, 0.6, 0.85, 1.0],
    labels=['Low risk', 'Medium risk', 'High risk'],
)

risk_summary = (
    scored.groupby('risk_group', observed=False)
    .agg(
        bookings=('actual_churn', 'count'),
        actual_churn_rate=('actual_churn', 'mean'),
        average_predicted_churn=('predicted_churn_probability', 'mean'),
    )
    .reset_index()
)
risk_summary

In [ ]:
plt.figure(figsize=(7, 4.5))
sns.barplot(data=risk_summary, x='risk_group', y='actual_churn_rate', color='#D64545')
plt.title('Actual churn by predicted risk group')
plt.xlabel('')
plt.ylabel('Actual churn rate')
plt.gca().yaxis.set_major_formatter(lambda x, _: f'{x:.0%}')
plt.tight_layout()
plt.show()

## 9. Leadership Summary and Recommendations

These are the points to carry into the PowerPoint deck.

In [ ]:
key_outputs = {
    'overall_churn_rate': df[TARGET].mean(),
    'roc_auc': roc_auc,
    'top_simple_factors': factor_importance_simple.head(5)['factor'].tolist(),
    'top_model_drivers': coef_table.head(8)['feature'].tolist(),
    'risk_summary': risk_summary.to_dict(orient='records'),
}
key_outputs

### Suggested PowerPoint Storyline

1. **Business context**: Churn is high, with around 44% of bookings not followed by another booking within six months.
2. **Main drivers**: Loyalty tier, customer type, marketing channel, platform, and booking/session behavior show the clearest relationship with churn.
3. **Model**: A logistic regression model can separate customers/bookings into more vs. less likely to churn groups.
4. **Risk groups**: High-risk bookings have materially higher observed churn than low-risk bookings.
5. **Recommendations**:
   - Prioritize retention actions for new customers and non-loyalty members.
   - Encourage loyalty sign-up and app/direct-channel engagement.
   - Review high-churn acquisition channels for retention quality, not only booking volume.
   - Use the model score to target retention offers, reminders, or personalized follow-ups.
   - Test interventions through A/B tests before scaling.

## 10. Save Core Analysis Outputs

In [ ]:
factor_importance_simple.to_csv(OUTPUT_DIR / 'factor_importance_simple.csv', index=False)
coef_table.to_csv(OUTPUT_DIR / 'logistic_regression_driver_importance.csv', index=False)
risk_summary.to_csv(OUTPUT_DIR / 'risk_group_summary.csv', index=False)
scored.to_csv(OUTPUT_DIR / 'scored_test_set.csv', index=False)

with open(OUTPUT_DIR / 'key_outputs.json', 'w') as f:
    json.dump(key_outputs, f, indent=2, default=str)

print(f'Outputs saved to: {OUTPUT_DIR}')